``AAPred().predict()`` scores raw sequences at three levels selected by ``level``: the whole protein (``'sequence'``), a domain with a boundary-sensitivity scan (``'domain'``), and a sliding per-residue window (``'window'``). We bind ``df_feat`` to the model so the feature matrix is computed internally from a ``df_seq`` (see [Breimann25]_):

``predict`` scores raw sequences; the ``level`` picks the granularity (one method, three cases):

- ``level='sequence'`` — **one score per protein** — *is this whole protein positive?*
- ``level='domain'`` — **boundary-sensitivity scan** — *how does the score depend on the exact boundary?*
- ``level='window'`` — **per-residue profile** — *which regions score highest?*

Add ``threshold=`` to also get a ``predicted_label`` column.

In [1]:
import aaanalysis as aa
aa.options["verbose"] = False  # Disable verbosity

# DOM_GSEC example dataset + its feature set (see [Breimann25]_)
df_seq = aa.load_dataset(name="DOM_GSEC")
labels = df_seq["label"].to_list()
df_feat = aa.load_features(name="DOM_GSEC").head(20)

# Training feature matrix
sf = aa.SequenceFeature()
X = sf.feature_matrix(features=df_feat, df_parts=sf.get_df_parts(df_seq=df_seq))

# Bind df_feat so raw sequences are featurized internally by predict()
aap = aa.AAPred(df_feat=df_feat, random_state=42)
aap.fit(X, labels)

**Sequence level** (``level='sequence'``). Each protein gets one positive-class score and its across-model standard deviation. ``list_parts`` selects which sequence parts are built for featurization (defaults to the standard CPP parts):

In [2]:
df_pred = aap.predict(df_seq.head(10), level="sequence",
                         list_parts=["tmd", "jmd_n_tmd_n", "tmd_c_jmd_c"])
aa.display_df(df_pred, n_rows=10, show_shape=True)

DataFrame shape: (10, 3)


,entry,score,score_std
1,P05067,0.960000,0.000000
2,P14925,0.940000,0.000000
3,P70180,0.970000,0.000000
4,Q03157,0.990000,0.000000
5,Q06481,1.000000,0.000000
6,P35613,0.940000,0.000000
7,P35070,0.630000,0.000000
8,P09803,0.990000,0.000000
9,P19022,1.000000,0.000000
10,P16070,0.930000,0.000000


Passing a ``threshold`` (in ``[0, 1]``) adds a ``predicted_label`` column (score at or above the threshold maps to the positive class, else the negative class):

In [3]:
df_pred = aap.predict(df_seq.head(10), level="sequence", threshold=0.5)
aa.display_df(df_pred, n_rows=10, show_shape=True)

DataFrame shape: (10, 4)


,entry,score,score_std,predicted_label
1,P05067,0.960000,0.000000,1
2,P14925,0.940000,0.000000,1
3,P70180,0.970000,0.000000,1
4,Q03157,0.990000,0.000000,1
5,Q06481,1.000000,0.000000,1
6,P35613,0.940000,0.000000,1
7,P35070,0.630000,0.000000,1
8,P09803,0.990000,0.000000,1
9,P19022,1.000000,0.000000,1
10,P16070,0.930000,0.000000,1


**Domain level** (``level='domain'``). The domain boundaries are shifted by every offset in ``[-window, +window]``; ``is_best`` flags the highest-scoring definition (offset ``0`` is the annotated boundary):

In [4]:
df_domain = aap.predict(df_seq[df_seq["entry"] == "P05067"], level="domain", window=3)
aa.display_df(df_domain, n_rows=10, show_shape=True)

DataFrame shape: (7, 4)


,entry,offset,score,is_best
1,P05067,-3,0.270000,False
2,P05067,-2,0.790000,False
3,P05067,-1,0.730000,False
4,P05067,0,0.960000,True
5,P05067,1,0.840000,False
6,P05067,2,0.530000,False
7,P05067,3,0.450000,False


**Window level** (``level='window'``). A length-``tmd_len`` window is slid along each sequence (spaced by ``step``), giving a per-residue profile. ``jmd_n_len`` / ``jmd_c_len`` set the flanking region required around each window:

In [5]:
row = df_seq[df_seq["entry"] == "P05067"].iloc[0]
tmd_len = int(row["tmd_stop"] - row["tmd_start"] + 1)
df_window = aap.predict(df_seq[df_seq["entry"] == "P05067"][["entry", "sequence"]],
                           level="window", tmd_len=tmd_len, step=5,
                           jmd_n_len=10, jmd_c_len=10)
aa.display_df(df_window, n_rows=10, show_shape=True)

DataFrame shape: (146, 4)


,entry,position,score,score_std
1,P05067,22,0.290000,0.000000
2,P05067,27,0.270000,0.000000
3,P05067,32,0.040000,0.000000
4,P05067,37,0.100000,0.000000
5,P05067,42,0.120000,0.000000
6,P05067,47,0.020000,0.000000
7,P05067,52,0.120000,0.000000
8,P05067,57,0.070000,0.000000
9,P05067,62,0.070000,0.000000
10,P05067,67,0.250000,0.000000
